# 03 · Student: distilled hierarchical open-set CNN (Colab GPU)

Trains the deployable student: one CNN forward pass → crop gate + per-crop cosine disease heads,
with **knowledge distillation** from the fused frozen teacher (notebook 02), **AM-Softmax** and **outlier exposure**.

Needs: dataset zips on Drive (notebook 01) and teacher features in `govi-ai/features/` (notebook 02).
Runtime → **T4 GPU**. One run of 30 epochs at 224 px ≈ 1–2 h on a T4 for MobileNetV4-S.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GOVI = '/content/drive/MyDrive/govi-ai'
DATA_ROOT = '/content/data'
FEATS = f'{GOVI}/features'
RUNS = f'{GOVI}/runs'

!git clone -q https://github.com/Kalana-Lakshan/govi-isuru-ai-service.git /content/govi-isuru-ai-service || git -C /content/govi-isuru-ai-service pull -q
%cd /content/govi-isuru-ai-service
!pip install -q -e ".[train]"
!python scripts/pack_dataset.py --unpack --out $GOVI/dataset --data-root $DATA_ROOT

## Main model (in-domain split)

In [ ]:
!python scripts/train_student.py --data-root $DATA_ROOT --features $FEATS --teacher dinov2_b clip_b16 \
    --student mnv4_s --protocol split --epochs 30 --out $RUNS/mnv4_s_split
!python scripts/eval_student.py --data-root $DATA_ROOT --run $RUNS/mnv4_s_split

## Ablations (paper table)
Each removes one component. Run them over several sessions; outputs land on Drive.

In [ ]:
ABLATIONS = {
    'no_kd':     '--teacher none',
    'no_oe':     '--w-oe 0',
    'no_margin': '--margin 0',
    'effnet_b0': '--student effnet_b0',
}
for name, flags in ABLATIONS.items():
    out = f'{RUNS}/mnv4_s_split_{name}'
    !python scripts/train_student.py --data-root $DATA_ROOT --features $FEATS --teacher dinov2_b clip_b16 --student mnv4_s --protocol split --epochs 30 --out {out} {flags}
    !python scripts/eval_student.py --data-root $DATA_ROOT --run {out}

## Cross-dataset (leave-one-dataset-out)

In [ ]:
import pandas as pd
lodo = [c for c in pd.read_csv(f'{DATA_ROOT}/manifest.csv', nrows=1).columns if c.startswith('lodo_')]
print(lodo)
for p in lodo:
    out = f'{RUNS}/mnv4_s_{p}'
    !python scripts/train_student.py --data-root $DATA_ROOT --features $FEATS --teacher dinov2_b clip_b16 --student mnv4_s --protocol {p} --epochs 20 --out {out}
    !python scripts/eval_student.py --data-root $DATA_ROOT --run {out}